In [1]:
"""
Diebold-Mariano Test for ResGCN Ablation Study
CORRECT IMPLEMENTATION with autocorrelation correction
Based on: Diebold & Mariano (1995) and Harvey, Leybourne & Newbold (1997)

References:
- Diebold, F.X. and R.S. Mariano (1995), "Comparing Predictive Accuracy," 
  Journal of Business & Economic Statistics, 13, 253-263.
- Harvey, D., S. Leybourne, and P. Newbold (1997), "Testing the Equality 
  of Prediction Mean Squared Errors," International Journal of Forecasting, 
  13(2), 281-291.
"""

import numpy as np
import pandas as pd
from pathlib import Path
from itertools import product
from scipy.stats import t as t_dist

# ===================== Configuration =====================
LOOKBACKS = [10, 22, 63]
HORIZONS = [1, 5, 10, 22, 63]
TARGET = "Aluminium"
ALL_FEATURES = ['Aluminium', 'Copper', 'Nickel', 'Zinc', 'Tin', 'Lead']

# Results directory (same as in the main script)
ROOT = Path("results_resgcn_aluminium_ablation_full")

# ===================== Helper Functions =====================

def newey_west_variance(d, h):
    """
    Newey-West HAC variance estimator for loss differential
    
    Parameters:
    -----------
    d : array-like
        Loss differential series
    h : int
        Forecast horizon (for determining lag length)
    
    Returns:
    --------
    variance : float
        HAC variance estimate
    """
    n = len(d)
    d_mean = np.mean(d)
    d_centered = d - d_mean
    
    # Gamma_0 (variance)
    gamma_0 = np.mean(d_centered ** 2)
    
    # Determine number of lags (common choice: h-1 for h-step forecasts)
    # But we use a more robust bandwidth selection
    max_lag = min(h - 1, int(np.floor(4 * (n / 100) ** (2/9))))
    max_lag = max(1, max_lag)  # At least 1 lag
    
    # Calculate autocovariances with Bartlett kernel weights
    gamma_sum = 0
    for k in range(1, max_lag + 1):
        # Autocovariance at lag k
        gamma_k = np.mean(d_centered[k:] * d_centered[:-k])
        
        # Bartlett kernel weight
        weight = 1 - k / (max_lag + 1)
        
        gamma_sum += 2 * weight * gamma_k
    
    # Long-run variance
    variance = gamma_0 + gamma_sum
    
    return max(variance, 1e-10)  # Ensure positive variance


def diebold_mariano_test(errors1, errors2, horizon, loss_power=2):
    """
    Diebold-Mariano test with Harvey-Leybourne-Newbold small sample correction
    
    Parameters:
    -----------
    errors1 : array-like
        Forecast errors from model 1 (reference model)
    errors2 : array-like
        Forecast errors from model 2 (comparison model)
    horizon : int
        Forecast horizon
    loss_power : int
        Power for loss function (1=MAE, 2=MSE)
    
    Returns:
    --------
    dm_stat : float
        DM test statistic
    p_value : float
        Two-tailed p-value
    """
    n = len(errors1)
    
    # Calculate loss differential
    # Loss function: |error|^power
    loss1 = np.abs(errors1) ** loss_power
    loss2 = np.abs(errors2) ** loss_power
    d = loss1 - loss2
    
    # Mean loss differential
    d_mean = np.mean(d)
    
    # HAC variance estimate using Newey-West
    d_var = newey_west_variance(d, horizon)
    
    # Standard DM statistic
    dm_stat = d_mean / np.sqrt(d_var / n)
    
    # Harvey-Leybourne-Newbold (1997) small sample correction
    # Modified statistic follows t-distribution with (n-1) degrees of freedom
    hlnb_correction = np.sqrt((n + 1 - 2 * horizon + horizon * (horizon - 1) / n) / n)
    dm_stat_corrected = dm_stat * hlnb_correction
    
    # Calculate p-value using t-distribution (more conservative for small samples)
    df = n - 1
    p_value = 2 * (1 - t_dist.cdf(abs(dm_stat_corrected), df))
    
    return dm_stat_corrected, p_value


def generate_input_sets(all_features, target):
    """Generate all possible input feature combinations"""
    from itertools import combinations
    
    others = [f for f in all_features if f != target]
    input_sets = {}
    
    for r in range(len(others) + 1):
        for combo in combinations(others, r):
            feats = [target] + list(combo)
            
            if len(feats) == len(all_features):
                tag = "FULL"
            else:
                excluded = sorted(set(others) - set(combo))
                tag = "w/o_" + "_".join(excluded)
            
            input_sets[tag] = feats
    
    return input_sets


# ===================== Main Processing =====================
def main():
    print("="*80)
    print("Diebold-Mariano Test with Newey-West HAC Variance")
    print("Based on Diebold & Mariano (1995) and Harvey et al. (1997)")
    print("="*80)
    print(f"\nRoot directory: {ROOT}")
    
    if not ROOT.exists():
        print(f"\n❌ Error: Results directory '{ROOT}' not found!")
        print("Please ensure the main training script has been run first.")
        return
    
    # Generate all input set tags
    INPUT_SETS = generate_input_sets(ALL_FEATURES, TARGET)
    
    dm_results = []
    reference_data = {}
    
    # ===================== Load Reference (FULL) Predictions =====================
    print("\n" + "="*80)
    print("STEP 1: Loading FULL model predictions (Reference)")
    print("="*80)
    
    for L, H in product(LOOKBACKS, HORIZONS):
        ref_dir = ROOT / f"FULL_LB{L}_H{H}"
        pred_file = ref_dir / "predictions_series.csv"
        
        if not pred_file.exists():
            print(f"⚠️  Warning: Missing FULL predictions for LB={L}, H={H}")
            continue
        
        df = pd.read_csv(pred_file)
        
        # Calculate errors
        errors = df['Actual'].values - df['Prediction'].values
        reference_data[(L, H)] = {
            'actual': df['Actual'].values,
            'prediction': df['Prediction'].values,
            'errors': errors
        }
        print(f"  ✅ Loaded FULL for LB={L:2d}, H={H:2d} ({len(errors):4d} samples)")
    
    # ===================== Compare All Models Against FULL =====================
    print("\n" + "="*80)
    print("STEP 2: Performing Diebold-Mariano Tests")
    print("="*80)
    print("\nFormat: Model Name | DM Statistic | p-value | Significance")
    print("  *** = p < 0.01, ** = p < 0.05")
    print("-"*80)
    
    total_tests = 0
    successful_tests = 0
    
    for L, H in product(LOOKBACKS, HORIZONS):
        print(f"\n📊 Lookback={L}, Horizon={H}")
        
        # Check if we have reference data
        if (L, H) not in reference_data:
            print(f"  ⚠️  Skipping - no FULL reference data")
            continue
        
        ref_errors = reference_data[(L, H)]['errors']
        
        # Test each input set variant
        for tag in sorted(INPUT_SETS.keys()):
            if tag == "FULL":
                continue  # Skip comparing FULL to itself
            
            total_tests += 1
            
            # Load comparison model predictions
            comp_dir = ROOT / f"{tag}_LB{L}_H{H}"
            pred_file = comp_dir / "predictions_series.csv"
            
            if not pred_file.exists():
                print(f"  ⚠️  Missing: {tag}")
                continue
            
            df_comp = pd.read_csv(pred_file)
            comp_errors = df_comp['Actual'].values - df_comp['Prediction'].values
            
            # Ensure same length
            if len(ref_errors) != len(comp_errors):
                print(f"  ⚠️  Length mismatch for {tag}: {len(ref_errors)} vs {len(comp_errors)}")
                continue
            
            # Perform DM test
            try:
                dm_stat, p_value = diebold_mariano_test(
                    ref_errors, 
                    comp_errors, 
                    horizon=H,
                    loss_power=2  # MSE loss
                )
                
                dm_results.append({
                    "Lookback": L,
                    "Horizon": H,
                    "Reference": "FULL",
                    "Compared_Model": tag,
                    "DM_stat": dm_stat,
                    "p_value": p_value,
                    "Significant_5pct": p_value < 0.05,
                    "Significant_1pct": p_value < 0.01
                })
                
                successful_tests += 1
                
                # Interpretation
                if dm_stat < 0:
                    interpretation = "FULL better"
                else:
                    interpretation = "Compared better"
                
                # Print result
                sig_marker = "***" if p_value < 0.01 else "**" if p_value < 0.05 else ""
                print(f"  ✅ {tag:25s} | DM={dm_stat:7.3f} | p={p_value:.4f} {sig_marker:3s} ({interpretation})")
                
            except Exception as e:
                print(f"  ❌ Error testing {tag}: {e}")
    
    # ===================== Save Results =====================
    if dm_results:
        df_dm = pd.DataFrame(dm_results)
        output_file = ROOT / "diebold_mariano_HAC_corrected.csv"
        df_dm.to_csv(output_file, index=False)
        
        print("\n" + "="*80)
        print("RESULTS SUMMARY")
        print("="*80)
        print(f"✅ Diebold-Mariano tests completed!")
        print(f"   Total tests attempted: {total_tests}")
        print(f"   Successful tests: {successful_tests}")
        print(f"   Results saved to: {output_file}")
        print("="*80)
        
        # Summary statistics
        sig_5 = df_dm['Significant_5pct'].sum()
        sig_1 = df_dm['Significant_1pct'].sum()
        
        print(f"\n📈 Statistical Significance:")
        print(f"   Significant at α=0.05: {sig_5}/{len(df_dm)} ({100*sig_5/len(df_dm):.1f}%)")
        print(f"   Significant at α=0.01: {sig_1}/{len(df_dm)} ({100*sig_1/len(df_dm):.1f}%)")
        
        # Direction of significance
        df_sig = df_dm[df_dm['Significant_5pct']]
        if len(df_sig) > 0:
            full_better = (df_sig['DM_stat'] < 0).sum()
            comp_better = (df_sig['DM_stat'] > 0).sum()
            print(f"\n   Among significant results:")
            print(f"   - FULL model better: {full_better}")
            print(f"   - Compared model better: {comp_better}")
        
        # Show most significant differences
        print("\n" + "="*80)
        print("TOP 10 MOST SIGNIFICANT DIFFERENCES")
        print("="*80)
        top10 = df_dm.nsmallest(10, 'p_value')[
            ['Lookback', 'Horizon', 'Compared_Model', 'DM_stat', 'p_value']
        ]
        print(top10.to_string(index=False))
        
        # Interpretation guide
        print("\n" + "="*80)
        print("INTERPRETATION GUIDE")
        print("="*80)
        print("DM Statistic:")
        print("  • Negative: FULL model has lower forecast error (FULL is better)")
        print("  • Positive: Compared model has lower forecast error (Compared is better)")
        print("\np-value:")
        print("  • < 0.01: Highly significant difference (***)")
        print("  • < 0.05: Significant difference (**)")
        print("  • ≥ 0.05: No significant difference")
        print("\nNote: Test uses Newey-West HAC variance with Harvey-Leybourne-Newbold")
        print("      small sample correction for robust inference.")
        print("="*80)
        
    else:
        print("\n⚠️  No DM test results generated. Check if predictions exist.")

if __name__ == "__main__":
    main()

Diebold-Mariano Test with Newey-West HAC Variance
Based on Diebold & Mariano (1995) and Harvey et al. (1997)

Root directory: results_resgcn_aluminium_ablation_full

STEP 1: Loading FULL model predictions (Reference)
  ✅ Loaded FULL for LB=10, H= 1 ( 684 samples)
  ✅ Loaded FULL for LB=10, H= 5 (3415 samples)
  ✅ Loaded FULL for LB=10, H=10 (6820 samples)
  ✅ Loaded FULL for LB=10, H=22 (14982 samples)
  ✅ Loaded FULL for LB=10, H=63 (42462 samples)
  ✅ Loaded FULL for LB=22, H= 1 ( 681 samples)
  ✅ Loaded FULL for LB=22, H= 5 (3405 samples)
  ✅ Loaded FULL for LB=22, H=10 (6810 samples)
  ✅ Loaded FULL for LB=22, H=22 (14938 samples)
  ✅ Loaded FULL for LB=22, H=63 (42399 samples)
  ✅ Loaded FULL for LB=63, H= 1 ( 676 samples)
  ✅ Loaded FULL for LB=63, H= 5 (3375 samples)
  ✅ Loaded FULL for LB=63, H=10 (6740 samples)
  ✅ Loaded FULL for LB=63, H=22 (14806 samples)
  ✅ Loaded FULL for LB=63, H=63 (42021 samples)

STEP 2: Performing Diebold-Mariano Tests

Format: Model Name | DM Stati